In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from tqdm import tqdm
from isds_tool.PS_data.zip_tools import uzip_file

In [3]:
# input_image_dir = r'/localnvme/data/added_data/test_data/test_data_mseg_c6_1021/images'
# input_label_dir = r'/localnvme/data/added_data/test_data/test_data_mseg_c6_1021/labels'
# output_label_dir = r'/localnvme/data/added_data/test_data/test_data_mseg_c6_1021_broken_refine/labels'
# check_image_dir = r'/localnvme/data/added_data/test_data/test_data_mseg_c6_1021/result_analysis/broken_refine'

input_image_dir = r'/localnvme/data/added_data/check1021/data29_check1021_mseg_c6/images'
input_label_dir = r'/localnvme/data/added_data/check1021/data29_check1021_mseg_c6/check1021_labels_1022_re'
output_label_dir = r'/localnvme/data/added_data/check1021/data29_check1021_mseg_c6_broken_refine/labels'
check_image_dir = r'/localnvme/data/added_data/check1021/data29_check1021_mseg_c6/result_analysis/broken_refine'
broken_dir_high = 'broken_high'
broken_dir_medium = 'broken_medium'
no_dir_list = [
    'no',
    'crack_high',
    'crack_medium',
    'hole_high',
    'hole_medium',
]

In [4]:
file_list = os.listdir(check_image_dir)
for file_name in file_list:
    if file_name.endswith('.zip'):
        file_path = os.path.join(check_image_dir, file_name)
        uzip_file(file_path, file_path.replace('.zip', ''))

In [5]:
def get_risk_refine_broken(check_image_dir, obj_image_name):
    broken_high_path = os.path.join(check_image_dir, broken_dir_high, obj_image_name)
    if os.path.exists(broken_high_path):
        return 2
    broken_medium_path = os.path.join(check_image_dir, broken_dir_medium, obj_image_name)
    if os.path.exists(broken_medium_path):
        return 1
    for no_dir_name in no_dir_list:
        no_dir_path = os.path.join(check_image_dir, no_dir_name, obj_image_name)
        if os.path.exists(no_dir_path):
            return 0
    return -1

def risk_refine_broken(input_image_dir, check_image_dir, input_gt_dir, output_gt_dir):
    os.makedirs(output_gt_dir, exist_ok=True)
    image_list = os.listdir(input_image_dir)
    broken_high_src_count, broken_medium_src_count, broken_no_src_count = 0, 0, 0
    broken_high_dst_count, broken_medium_dst_count, broken_no_dst_count = 0, 0, 0
    for image_name in tqdm(image_list):
        label_name = Path(image_name).stem + '.txt'
        input_gt_path = os.path.join(input_gt_dir, label_name)
        output_gt_path = os.path.join(output_gt_dir, label_name)
        with open(input_gt_path, 'r') as fi, open(output_gt_path, 'w') as fo:
            data = fi.readlines()
            new_data = []
            for id_line, line in enumerate(data):
                parts = line.strip().split(' ')
                category = int(parts[0])
                att_len = int(parts[1])
                atts = list(map(int, parts[2:2 + att_len]))
                polygons = list(map(float, parts[2 + att_len:]))

                risk_broken = atts[1]
                if risk_broken > 0:
                    if risk_broken == 1:
                        broken_medium_src_count += 1
                    else:
                        broken_high_src_count += 1
                    obj_image_name = Path(image_name).stem + f'_{id_line}' + Path(image_name).suffix
                    obj_result = get_risk_refine_broken(check_image_dir, obj_image_name)
                    if obj_result == 0:
                        broken_no_dst_count += 1
                    elif obj_result == 1:
                        broken_medium_dst_count += 1
                    elif obj_result == 2:
                        broken_high_dst_count += 1
                    else:
                        print(f'{obj_image_name} cannot find!')
                        obj_result = risk_broken
                    atts[1] = obj_result
                info = [category, att_len] + atts + polygons
                new_line = ' '.join(map(str, info)) +'\n'
                new_data.append(new_line)
            fo.writelines(new_data)


In [6]:
risk_refine_broken(input_image_dir, check_image_dir, input_label_dir, output_label_dir)

100%|██████████| 29/29 [00:00<00:00, 1074.84it/s]
